# Evaluate Performance Using Multi Layer Perceptron

In [1]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [3]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,swir16,swir22,NDMI,MNDWI,cec,clay,pH,phosphorous,flow_accumulation,skin_temperature,soil_temperature,temperature_2m,total_evaporation_sum,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,24.0,21.0,75.0,23.0,4.131443e+06,307.919884,307.833471,300.198610,-0.000085,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,27.0,29.0,65.0,21.0,1.162031e+04,293.665109,294.393847,292.444443,-0.004103,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,23.0,28.0,64.0,21.0,1.000000e+00,293.972931,294.725525,292.828033,-0.004010,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,23.0,26.0,65.0,22.0,1.878500e+04,294.280160,295.166729,293.509254,-0.003895,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,25.0,28.0,64.0,20.0,4.778000e+03,294.414705,295.304085,293.890963,-0.003909,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [4]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [5]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,swir16,swir22,NDMI,MNDWI,cec,clay,pH,phosphorous,flow_accumulation,skin_temperature,soil_temperature,temperature_2m,total_evaporation_sum,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
8222,218.563,188.1,41.0,213.10000,7.506967,3457.0,5727.0,15389.5,NaN,NaN,NaN,NaN,NaN,NaN,19.0,21.0,70.0,23.0,250775.000000,296.419816,297.277244,293.422777,-0.000151,0.000023,0.056819,0.000000,1330.0,1610.0,88135666.5,5.266275e+06,5.767825e+06,-6.426571,0.904762
7382,62.377,189.1,20.0,162.90001,244.994357,4085.5,6344.0,15206.0,16907.5,10034.0,15675.5,13225.0,0.037811,-0.219433,25.0,23.0,66.0,25.0,9632.878049,300.602034,300.912767,299.292981,-0.003207,0.000668,0.289246,101.878327,1650.0,1650.0,96466864.0,2.215562e+05,2.408220e+05,-4.794787,1.086956
3398,209.011,806.0,18.0,169.10000,1281.373723,3535.0,5953.0,15300.0,16558.5,9783.0,13556.0,11094.5,0.099703,-0.161661,25.0,26.0,64.0,19.0,30946.000000,297.658065,298.210755,296.360579,-0.002824,0.001285,0.192747,57.174639,1600.0,1216.0,91080900.0,8.045960e+05,5.879740e+05,-2.195708,0.961538
1211,102.667,371.0,24.0,143.80000,601.159570,3805.0,6506.0,14973.0,16235.0,9879.5,13432.0,10823.0,0.094482,-0.152393,24.0,28.0,59.0,25.0,2154.357143,288.531571,289.381872,287.944758,-0.002494,0.001500,0.399498,45.611303,1416.0,1475.0,97414338.0,6.032200e+04,5.385893e+04,-1.661581,0.857143
26,82.432,283.8,32.0,173.20000,1363.117917,4003.0,6415.0,15005.0,18631.5,11771.5,16495.0,12815.0,0.060823,-0.167106,20.0,23.0,61.0,19.0,791.000000,292.901982,293.745146,291.850964,-0.003821,0.005010,0.371570,44.405529,1220.0,1159.0,96257075.0,1.819300e+04,1.502900e+04,-0.762548,0.869565


In [6]:
train_df.shape

(6523, 33)

In [7]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## **Data Preprocessing** 